In [19]:
import json
import requests

import torch
import spacy

from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

nlp = spacy.load("en_core_web_sm")

print(device)

cuda


In [21]:
MODEL_PATH = "./flan_t5_mintaka/checkpoint-7000"

# Tokenizer comes from the original Flan-T5 model
tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-base"
)

# Fine-tuned weights come from your checkpoint
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATH
)

model = model.to(device)

model.eval()

print("Fine-tuned Flan-T5 checkpoint loaded successfully!")

Fine-tuned Flan-T5 checkpoint loaded successfully!


In [22]:
test_prompt = """
Answer the question.

Question:
What is the capital of France?

Answer:
"""

inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=10
    )

print(
    tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )
)

london


In [23]:
q = "What is the capital of France?"

prompt = "question: " + q

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=128
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=32,
        num_beams=4
    )

answer = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("Question:", q)
print("Prediction:", answer)

Question: What is the capital of France?
Prediction: london


In [24]:
q = test_q[0]

print("Question:", q)
print("Gold:", test_a[0])

prompt = "question: " + q

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=128
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=32,
        num_beams=4
    )

prediction = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("Prediction:", prediction)

Question: What man was a famous American author and also a steamboat pilot on the Mississippi River?
Gold: Mark Twain
Prediction: charles dickens


In [25]:
q = test_q[0]

prompt = f"""
You are an intelligent question answering assistant.

Carefully understand the question.
Reason through the facts internally before producing the answer.

Question:
{q}

Final Answer:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=32,
        num_beams=4
    )

prediction = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("Question:", q)
print("Gold:", test_a[0])
print("Prediction:", prediction)

Question: What man was a famous American author and also a steamboat pilot on the Mississippi River?
Gold: Mark Twain
Prediction: Thomas Jefferson.


In [26]:
# ============================================================
# DIAGNOSTIC: TEST FINE-TUNED FLAN-T5
# ============================================================

import torch
from tqdm import tqdm

model.eval()

def predict_ft5(question):
    
    prompt = f"""
You are an intelligent question answering assistant.

Carefully understand the question.
Reason through the facts internally before producing the answer.

Question:
{question}

Final Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()


print("=" * 70)
print("FINE-TUNED FLAN-T5 DIAGNOSTIC")
print("=" * 70)

for i in range(5):

    q = test_q[i]
    gold = test_a[i]
    pred = predict_ft5(q)

    print("\nQuestion:", q)
    print("Gold:", gold)
    print("Prediction:", pred)
    print("-" * 70)

FINE-TUNED FLAN-T5 DIAGNOSTIC

Question: What man was a famous American author and also a steamboat pilot on the Mississippi River?
Gold: Mark Twain
Prediction: Thomas Jefferson.
----------------------------------------------------------------------

Question: How many Academy Awards has Jake Gyllenhaal been nominated for?
Gold: 1
Prediction: three
----------------------------------------------------------------------

Question: Who is older, The Weeknd or Drake?
Gold: Drake
Prediction: Drake
----------------------------------------------------------------------

Question: How many children did Donald Trump have?
Gold: 5
Prediction: Donald Trump has two children.
----------------------------------------------------------------------

Question: Is the main hero in Final Fantasy IX named Kuja?
Gold: False
Prediction: Kuja is a character in Final Fantasy IX.
----------------------------------------------------------------------


In [27]:
# ============================================================
# TEST A TRAINING EXAMPLE
# ============================================================

for i in [0, 100, 500, 1000]:

    q = train_q[i]
    gold = train_a[i]
    pred = predict_ft5(q)

    print("\nQuestion:", q)
    print("Gold:", gold)
    print("Prediction:", pred)
    print("=" * 70)

NameError: name 'train_q' is not defined

In [28]:
import json
import os

def extract_answer(item):
    ans = item.get("answer")

    if not ans:
        return ""

    if isinstance(ans, dict):
        arr = ans.get("answer")

        if not arr:
            return ""

        if isinstance(arr, list):
            values = []

            for obj in arr:
                if isinstance(obj, dict):
                    label = obj.get("label", "")

                    if isinstance(label, dict):
                        label = label.get("en", "")

                    if label:
                        values.append(str(label))
                else:
                    values.append(str(obj))

            return ", ".join(values)

        return str(arr)

    return str(ans)


def load_mintaka(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    questions = []
    answers = []

    for item in data:
        q = item.get("question", "").strip()
        a = extract_answer(item).strip()

        if q and a:
            questions.append(q)
            answers.append(a)

    return questions, answers


train_q, train_a = load_mintaka("mintaka_train.json")
dev_q, dev_a = load_mintaka("mintaka_dev.json")
test_q, test_a = load_mintaka("mintaka_test.json")

print("Train:", len(train_q))
print("Dev:", len(dev_q))
print("Test:", len(test_q))

Train: 13791
Dev: 1981
Test: 3933


In [29]:
!pip install -q scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer

retriever = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=50000,
    sublinear_tf=True
)

train_vectors = retriever.fit_transform(train_q)

print("Retrieval index created")
print("Shape:", train_vectors.shape)

Retrieval index created
Shape: (13791, 40316)


In [31]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve_context(question, top_k=5):

    query_vector = retriever.transform([question])

    scores = cosine_similarity(
        query_vector,
        train_vectors
    )[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    retrieved = []

    for idx in top_indices:
        retrieved.append({
            "question": train_q[idx],
            "answer": train_a[idx],
            "score": float(scores[idx])
        })

    return retrieved

In [32]:
q = test_q[0]

results = retrieve_context(q, top_k=5)

print("QUERY:")
print(q)

print("\nRETRIEVED CONTEXT:")
print("=" * 70)

for i, item in enumerate(results, 1):

    print(f"\n{i}. Score: {item['score']:.4f}")
    print("Question:", item["question"])
    print("Answer:", item["answer"])

QUERY:
What man was a famous American author and also a steamboat pilot on the Mississippi River?

RETRIEVED CONTEXT:

1. Score: 0.3993
Question: Who was a famous American author and an oyster pirate in his youth?
Answer: Jack London

2. Score: 0.3787
Question: Which river is longer than the Mississippi River?
Answer: Nile

3. Score: 0.3698
Question: Is the Mississippi River the longest river in the world?
Answer: False

4. Score: 0.3216
Question: Is the Mississippi River located in the United States?
Answer: True

5. Score: 0.3198
Question: Is the Missouri River longer than the Mississippi River?
Answer: True


In [33]:
!pip install -q sentence-transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [34]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

print("Embedding model loaded")

Embedding model loaded


In [35]:
train_embeddings = embedding_model.encode(
    train_q,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Training embeddings created")
print("Shape:", train_embeddings.shape)

Batches: 100%|██████████| 216/216 [00:01<00:00, 187.70it/s]

Training embeddings created
Shape: (13791, 384)


In [36]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def semantic_retrieve(question, top_k=5):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embedding,
        train_embeddings
    )[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:

        results.append({
            "question": train_q[idx],
            "answer": train_a[idx],
            "score": float(scores[idx])
        })

    return results

In [37]:
q = "What man was a famous American author and also a steamboat pilot on the Mississippi River?"

results = semantic_retrieve(
    q,
    top_k=5
)

print("QUERY:")
print(q)

print("\nSEMANTIC RETRIEVED CONTEXT:")
print("=" * 80)

for i, item in enumerate(results, 1):

    print(f"\n{i}. Score: {item['score']:.4f}")
    print("Question:", item["question"])
    print("Answer:", item["answer"])

QUERY:
What man was a famous American author and also a steamboat pilot on the Mississippi River?

SEMANTIC RETRIEVED CONTEXT:

1. Score: 0.5662
Question: Is Tom Sawyer the name of an author?
Answer: False

2. Score: 0.5236
Question: Who was a famous American author and an oyster pirate in his youth?
Answer: Jack London

3. Score: 0.5046
Question: Author who has written the most books?
Answer: Ryoki Inoue

4. Score: 0.4943
Question: Who is the first American female author that was published?
Answer: Anne Bradstreet

5. Score: 0.4789
Question: Which river did Huckleberry Finn travel?
Answer: Mississippi River


In [38]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re


def keyword_similarity(question, candidate):

    q_tokens = set(
        re.findall(
            r"\b[a-zA-Z0-9]+\b",
            question.lower()
        )
    )

    c_tokens = set(
        re.findall(
            r"\b[a-zA-Z0-9]+\b",
            candidate.lower()
        )
    )

    if not q_tokens or not c_tokens:
        return 0.0

    overlap = q_tokens.intersection(c_tokens)

    return len(overlap) / len(q_tokens)

In [39]:
def hybrid_retrieve(question, top_k=5):

    # Semantic retrieval
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    semantic_scores = cosine_similarity(
        query_embedding,
        train_embeddings
    )[0]

    # Retrieve a larger candidate pool first
    candidate_indices = np.argsort(
        semantic_scores
    )[::-1][:50]

    candidates = []

    for idx in candidate_indices:

        semantic_score = semantic_scores[idx]

        keyword_score = keyword_similarity(
            question,
            train_q[idx]
        )

        # Weighted combination
        combined_score = (
            0.75 * semantic_score +
            0.25 * keyword_score
        )

        candidates.append({
            "index": idx,
            "question": train_q[idx],
            "answer": train_a[idx],
            "semantic_score": float(
                semantic_score
            ),
            "keyword_score": float(
                keyword_score
            ),
            "score": float(
                combined_score
            )
        })

    candidates.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return candidates[:top_k]

In [40]:
q = "What man was a famous American author and also a steamboat pilot on the Mississippi River?"

results = hybrid_retrieve(
    q,
    top_k=5
)

print("QUERY:")
print(q)

print("\nHYBRID RETRIEVAL:")
print("=" * 80)

for i, item in enumerate(results, 1):

    print(f"\n{i}. Combined Score: {item['score']:.4f}")
    print(
        f"   Semantic: {item['semantic_score']:.4f}"
    )
    print(
        f"   Keyword: {item['keyword_score']:.4f}"
    )

    print("   Question:", item["question"])
    print("   Answer:", item["answer"])

QUERY:
What man was a famous American author and also a steamboat pilot on the Mississippi River?

HYBRID RETRIEVAL:

1. Combined Score: 0.4927
   Semantic: 0.5236
   Keyword: 0.4000
   Question: Who was a famous American author and an oyster pirate in his youth?
   Answer: Jack London

2. Combined Score: 0.4580
   Semantic: 0.5662
   Keyword: 0.1333
   Question: Is Tom Sawyer the name of an author?
   Answer: False

3. Combined Score: 0.4374
   Semantic: 0.4943
   Keyword: 0.2667
   Question: Who is the first American female author that was published?
   Answer: Anne Bradstreet

4. Combined Score: 0.4170
   Semantic: 0.4449
   Keyword: 0.3333
   Question: Who was the first African American governor and a Democrat?
   Answer: Douglas Wilder

5. Combined Score: 0.4168
   Semantic: 0.4446
   Keyword: 0.3333
   Question: When was the author who wrote The Old Man and the Sea born?
   Answer: 18.99


In [41]:
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_entities(question):

    doc = nlp(question)

    entities = []

    for ent in doc.ents:
        if ent.label_ in [
            "PERSON",
            "ORG",
            "GPE",
            "LOC",
            "FAC",
            "WORK_OF_ART",
            "EVENT",
            "PRODUCT"
        ]:
            entities.append(ent.text)

    # Remove duplicates
    entities = list(dict.fromkeys(entities))

    return entities

In [42]:
q = "What man was a famous American author and also a steamboat pilot on the Mississippi River?"

entities = extract_entities(q)

print("Question:")
print(q)

print("\nEntities:")
print(entities)

Question:
What man was a famous American author and also a steamboat pilot on the Mississippi River?

Entities:
['the Mississippi River']


In [43]:
import requests

HEADERS = {
    "User-Agent": "Mintaka-KGQA-Research/1.0"
}

wikidata_cache = {}

def wikidata_search(entity, limit=5):

    if entity in wikidata_cache:
        return wikidata_cache[entity]

    url = "https://www.wikidata.org/w/api.php"

    params = {
        "action": "wbsearchentities",
        "search": entity,
        "language": "en",
        "format": "json",
        "limit": limit
    }

    try:

        response = requests.get(
            url,
            params=params,
            headers=HEADERS,
            timeout=20
        )

        response.raise_for_status()

        data = response.json()

        results = []

        for item in data.get("search", []):

            results.append({
                "id": item.get("id"),
                "label": item.get("label", ""),
                "description": item.get(
                    "description", ""
                )
            })

        wikidata_cache[entity] = results

        return results

    except Exception as e:

        print("Wikidata error:", e)
        return []

In [44]:
def get_wikidata_candidates(question):

    entities = extract_entities(question)

    candidates = []

    for entity in entities:

        results = wikidata_search(
            entity,
            limit=5
        )

        for result in results:

            candidates.append({
                "mention": entity,
                "id": result["id"],
                "label": result["label"],
                "description": result["description"]
            })

    return candidates

In [45]:
q = "What man was a famous American author and also a steamboat pilot on the Mississippi River?"

candidates = get_wikidata_candidates(q)

for i, item in enumerate(candidates, 1):

    print(f"\n{i}.")
    print("Mention:", item["mention"])
    print("ID:", item["id"])
    print("Label:", item["label"])
    print("Description:", item["description"])


1.
Mention: the Mississippi River
ID: Q50388374
Label: The Mississippi River Problem
Description: article in the July 1908 issue of the Popular Science Monthly

2.
Mention: the Mississippi River
ID: Q137103851
Label: The Mississippi River
Description: 

3.
Mention: the Mississippi River
ID: Q113061049
Label: The Mississippi River
Description: scientific article published on 15 December 2016

4.
Mention: the Mississippi River
ID: Q64328503
Label: The Mississippi River
Description: 

5.
Mention: the Mississippi River
ID: Q61806559
Label: The Mississippi River records glacial-isostatic deformation of North America
Description: scientific article published on 30 January 2019


In [46]:
fact_cache = {}

def get_wikidata_facts(entity_id):

    if entity_id in fact_cache:
        return fact_cache[entity_id]

    url = f"https://www.wikidata.org/wiki/Special:EntityData/{entity_id}.json"

    try:

        response = requests.get(
            url,
            headers=HEADERS,
            timeout=20
        )

        response.raise_for_status()

        entity = response.json()[
            "entities"
        ][entity_id]

        labels = entity.get(
            "labels", {}
        )

        label = labels.get(
            "en",
            {}
        ).get(
            "value",
            ""
        )

        descriptions = entity.get(
            "descriptions",
            {}
        )

        description = descriptions.get(
            "en",
            {}
        ).get(
            "value",
            ""
        )

        facts = []

        if label:
            facts.append(
                f"Entity: {label}"
            )

        if description:
            facts.append(
                f"Description: {description}"
            )

        fact_cache[entity_id] = facts

        return facts

    except Exception as e:

        print("Fact retrieval error:", e)
        return []

In [47]:
def build_kg_context(question):

    candidates = get_wikidata_candidates(
        question
    )

    context = []

    for item in candidates[:10]:

        facts = get_wikidata_facts(
            item["id"]
        )

        context.extend(facts)

    # Remove duplicates
    context = list(
        dict.fromkeys(context)
    )

    return context

In [48]:
q = "What man was a famous American author and also a steamboat pilot on the Mississippi River?"

kg_context = build_kg_context(q)

print("KG CONTEXT")
print("=" * 80)

for fact in kg_context:

    print(fact)

KG CONTEXT
Entity: The Mississippi River Problem
Description: article in the July 1908 issue of the Popular Science Monthly
Entity: The Mississippi River
Description: scientific article published on 15 December 2016
Entity: The Mississippi River records glacial-isostatic deformation of North America
Description: scientific article published on 30 January 2019


In [49]:
def load_mintaka_full(path):

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    records = []

    for item in data:

        answer_obj = item.get("answer")

        if not answer_obj:
            continue

        answer_mention = answer_obj.get(
            "mention",
            ""
        )

        if not answer_mention:
            continue

        entities = []

        for ent in item.get(
            "questionEntity",
            []
        ):

            label = ent.get(
                "label",
                ""
            )

            if isinstance(label, dict):
                label = label.get(
                    "en",
                    ""
                )

            mention = ent.get(
                "mention",
                ""
            )

            if label:
                entities.append(label)

            elif mention:
                entities.append(mention)

        supporting_entities = []

        for ent in answer_obj.get(
            "supportingEnt",
            []
        ):

            label = ent.get(
                "label",
                {}
            )

            if isinstance(label, dict):
                label = label.get(
                    "en",
                    ""
                )

            if label:
                supporting_entities.append(
                    label
                )

        records.append({
            "question": item["question"],
            "answer": answer_mention,
            "entities": entities,
            "supporting_entities":
                supporting_entities,
            "category":
                item.get("category", ""),
            "complexity":
                item.get(
                    "complexityType",
                    ""
                )
        })

    return records

In [50]:
train_records = load_mintaka_full(
    "mintaka_train.json"
)

print(
    "Training records:",
    len(train_records)
)

print("\nExample record:")

print(
    json.dumps(
        train_records[0],
        indent=2,
        ensure_ascii=False
    )
)

Training records: 14000

Example record:
{
  "question": "What is the seventh tallest mountain in North America?",
  "answer": "Mount Lucania",
  "entities": [
    "North America",
    "seventh"
  ],
  "supporting_entities": [],
  "category": "geography",
  "complexity": "ordinal"
}


In [51]:
train_record_questions = [
    r["question"]
    for r in train_records
]

train_record_embeddings = (
    embedding_model.encode(
        train_record_questions,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
)

print(
    "Embedding shape:",
    train_record_embeddings.shape
)

Batches: 100%|██████████| 219/219 [00:01<00:00, 158.88it/s]

Embedding shape: (14000, 384)


In [52]:
def retrieve_entity_aware_context(
    question,
    top_k=5
):

    query_embedding = (
        embedding_model.encode(
            [question],
            convert_to_numpy=True,
            normalize_embeddings=True
        )
    )

    scores = cosine_similarity(
        query_embedding,
        train_record_embeddings
    )[0]

    top_indices = np.argsort(
        scores
    )[::-1][:top_k]

    results = []

    for idx in top_indices:

        record = train_records[idx]

        results.append({
            "question":
                record["question"],

            "answer":
                record["answer"],

            "entities":
                record["entities"],

            "supporting_entities":
                record[
                    "supporting_entities"
                ],

            "category":
                record["category"],

            "complexity":
                record["complexity"],

            "score":
                float(scores[idx])
        })

    return results

In [53]:
q = (
    "What man was a famous American author "
    "and also a steamboat pilot on the "
    "Mississippi River?"
)

results = retrieve_entity_aware_context(
    q,
    top_k=5
)

print("QUESTION:")
print(q)

print("\nENTITY-AWARE RETRIEVAL")
print("=" * 80)

for i, item in enumerate(results, 1):

    print(f"\n{i}. Score: {item['score']:.4f}")

    print(
        "Question:",
        item["question"]
    )

    print(
        "Answer:",
        item["answer"]
    )

    print(
        "Entities:",
        item["entities"]
    )

    print(
        "Supporting entities:",
        item["supporting_entities"]
    )

    print(
        "Category:",
        item["category"]
    )

    print(
        "Complexity:",
        item["complexity"]
    )

QUESTION:
What man was a famous American author and also a steamboat pilot on the Mississippi River?

ENTITY-AWARE RETRIEVAL

1. Score: 0.5662
Question: Is Tom Sawyer the name of an author?
Answer: No
Entities: ['Tom Sawyer']
Supporting entities: []
Category: books
Complexity: yesno

2. Score: 0.5236
Question: Who was a famous American author and an oyster pirate in his youth?
Answer: Jack London
Entities: ['Americans']
Supporting entities: []
Category: history
Complexity: intersection

3. Score: 0.5046
Question: Author who has written the most books?
Answer: Ryoki Inoue
Entities: ['writer', 'book']
Supporting entities: []
Category: books
Complexity: superlative

4. Score: 0.4943
Question: Who is the first American female author that was published?
Answer: Anne Bradstreet
Entities: ['Americans', 'first']
Supporting entities: []
Category: books
Complexity: generic

5. Score: 0.4789
Question: Which river did Huckleberry Finn travel?
Answer: Mississippi River
Entities: ['Huckleberry Finn'

In [54]:
!pip install -q sentence-transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [55]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device=device
)

print("Cross-Encoder reranker loaded")

Cross-Encoder reranker loaded


In [56]:
def retrieve_candidates(question, candidate_k=50):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embedding,
        train_record_embeddings
    )[0]

    top_indices = np.argsort(
        scores
    )[::-1][:candidate_k]

    candidates = []

    for idx in top_indices:

        candidates.append({
            "index": int(idx),
            "question": train_records[idx]["question"],
            "answer": train_records[idx]["answer"],
            "entities": train_records[idx]["entities"],
            "supporting_entities":
                train_records[idx]["supporting_entities"],
            "semantic_score": float(scores[idx])
        })

    return candidates

In [57]:
def rerank_candidates(
    question,
    candidates,
    top_k=5
):

    pairs = [
        (
            question,
            candidate["question"]
        )
        for candidate in candidates
    ]

    rerank_scores = reranker.predict(
        pairs,
        show_progress_bar=False
    )

    for candidate, score in zip(
        candidates,
        rerank_scores
    ):
        candidate["rerank_score"] = float(score)

    candidates.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return candidates[:top_k]

In [58]:
def retrieve_with_reranking(
    question,
    candidate_k=50,
    top_k=5
):

    candidates = retrieve_candidates(
        question,
        candidate_k=candidate_k
    )

    results = rerank_candidates(
        question,
        candidates,
        top_k=top_k
    )

    return results

In [59]:
q = (
    "What man was a famous American author "
    "and also a steamboat pilot on the "
    "Mississippi River?"
)

results = retrieve_with_reranking(
    q,
    candidate_k=50,
    top_k=5
)

print("QUESTION:")
print(q)

print("\nRERANKED RESULTS")
print("=" * 80)

for i, item in enumerate(results, 1):

    print(f"\n{i}.")
    print("Rerank score:",
          round(item["rerank_score"], 4))

    print("Semantic score:",
          round(item["semantic_score"], 4))

    print("Question:",
          item["question"])

    print("Answer:",
          item["answer"])

    print("Entities:",
          item["entities"])

QUESTION:
What man was a famous American author and also a steamboat pilot on the Mississippi River?

RERANKED RESULTS

1.
Rerank score: -1.8074
Semantic score: 0.5236
Question: Who was a famous American author and an oyster pirate in his youth?
Answer: Jack London
Entities: ['Americans']

2.
Rerank score: -5.4148
Semantic score: 0.4407
Question: Who was the famous poet and author who was in the West Point class of 1834, but never graduated?
Answer: Edgar Allen Poe
Entities: ['United States Military Academy', '1834']

3.
Rerank score: -6.0052
Semantic score: 0.4693
Question: Is the Mississippi River located in the United States?
Answer: Yes
Entities: ['Mississippi River', 'United States of America']

4.
Rerank score: -6.0978
Semantic score: 0.4653
Question: How many books did author M. Williams Phelps write?
Answer: 52
Entities: ['M. William Phelps']

5.
Rerank score: -6.1281
Semantic score: 0.4679
Question: Who is the oldest known author in history?
Answer: Enheduanna
Entities: ['writ

In [60]:
def build_reranked_context(
    results
):

    context_parts = []

    for i, item in enumerate(
        results,
        1
    ):

        context_parts.append(
            f"""
Evidence {i}:
Question: {item["question"]}
Answer: {item["answer"]}
Entities: {", ".join(item["entities"])}
"""
        )

    return "\n".join(
        context_parts
    )

In [61]:
def build_reranked_rag_prompt(
    question,
    results
):

    context = build_reranked_context(
        results
    )

    prompt = f"""
You are an intelligent question answering assistant.

Carefully understand the question and use
the retrieved evidence when it is relevant.

Retrieved Evidence:
{context}

Question:
{question}

Final Answer:
"""

    return prompt

In [62]:
def predict_reranked_rag(
    question,
    candidate_k=50,
    top_k=5
):

    results = retrieve_with_reranking(
        question,
        candidate_k=candidate_k,
        top_k=top_k
    )

    prompt = build_reranked_rag_prompt(
        question,
        results
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            num_beams=5,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    return prediction, results

In [63]:
for i in range(10):

    q = test_q[i]
    gold = test_a[i]

    prediction, results = (
        predict_reranked_rag(q)
    )

    print("\n" + "=" * 80)
    print("Question:", q)
    print("Gold:", gold)
    print("Prediction:", prediction)

    print("\nTop Retrieved Evidence:")

    for j, item in enumerate(
        results,
        1
    ):
        print(
            f"{j}. "
            f"{item['answer']} "
            f"({item['rerank_score']:.3f})"
        )


Question: What man was a famous American author and also a steamboat pilot on the Mississippi River?
Gold: Mark Twain
Prediction: William Phelps

Top Retrieved Evidence:
1. Jack London (-1.807)
2. Edgar Allen Poe (-5.415)
3. Yes (-6.005)
4. 52 (-6.098)
5. Enheduanna (-6.128)

Question: How many Academy Awards has Jake Gyllenhaal been nominated for?
Gold: 1
Prediction: 2

Top Retrieved Evidence:
1. 1 (-0.775)
2. Yes (-1.322)
3. 3 (-1.387)
4. 7 (-1.767)
5. Meryl Streep (-1.827)

Question: Who is older, The Weeknd or Drake?
Gold: Drake
Prediction: Drake

Top Retrieved Evidence:
1. Eminem (4.558)
2. Kanye West (1.754)
3. Drake (-0.260)
4. Drake (-0.863)
5. Rihanna (-3.148)

Question: How many children did Donald Trump have?
Gold: 5
Prediction: 4

Top Retrieved Evidence:
1. 3 (0.803)
2. 3 (-0.149)
3. 6 (-0.240)
4. Four (Arabella, Caroline, John Jr., Patrick) (-0.501)
5. 2 (-1.309)

Question: Is the main hero in Final Fantasy IX named Kuja?
Gold: False
Prediction: Entities: Final Fantasy IX

In [64]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [65]:
def retrieve_rag_candidates(question, top_k=5):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embedding,
        train_embeddings
    )[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    candidates = []

    for idx in top_indices:

        candidates.append({
            "question": train_q[idx],
            "answer": train_a[idx],
            "score": float(scores[idx])
        })

    return candidates

In [66]:
def build_rag_context(candidates):

    context = []

    for i, item in enumerate(candidates, 1):

        context.append(
            f"Example {i}: "
            f"Question: {item['question']} "
            f"Answer: {item['answer']}"
        )

    return "\n".join(context)

In [78]:
def build_rag_prompt(question, candidates):

    context = build_rag_context(candidates)

    prompt = f"""
You are an intelligent question answering assistant.

Carefully understand the question.
Use the retrieved knowledge as supporting evidence.
Reason through the facts internally before producing the answer.

Retrieved Knowledge:
{context}

Question:
{question}

Final Answer:
"""

    return prompt

In [80]:
def predict_rag(question, top_k=5):

    candidates = retrieve_rag_candidates(
        question,
        top_k=top_k
    )

    prompt = build_rag_prompt(
        question,
        candidates
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            num_beams=5,
            early_stopping=True
        )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    return prediction, candidates

In [79]:
def predict_baseline(question):

    prompt = f"""
You are an intelligent question answering assistant.

Carefully understand the question.
Reason through the facts internally before producing the answer.

Question:
{question}

Final Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            num_beams=5,
            early_stopping=True
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

In [70]:
def hybrid_prediction(question):

    baseline = predict_baseline(question)

    rag_prediction, candidates = predict_rag(
        question,
        top_k=5
    )

    # Highest retrieval confidence
    best_score = candidates[0]["score"]

    # Use RAG only when retrieval is reasonably strong
    if best_score >= 0.65:

        final_prediction = rag_prediction
        method = "RAG"

    else:

        final_prediction = baseline
        method = "Baseline fallback"

    return (
        final_prediction,
        baseline,
        rag_prediction,
        method,
        candidates
    )

In [81]:
for i in range(20):

    q = test_q[i]
    gold = test_a[i]

    baseline = predict_baseline(q)

    rag_prediction, candidates = predict_rag(
        q,
        top_k=5
    )

    print("\n" + "=" * 80)

    print("Question:", q)
    print("Gold:", gold)
    print("Baseline:", baseline)
    print("RAG:", rag_prediction)


Question: What man was a famous American author and also a steamboat pilot on the Mississippi River?
Gold: Mark Twain
Baseline: Thomas Jefferson.
RAG: Jack London

Question: How many Academy Awards has Jake Gyllenhaal been nominated for?
Gold: 1
Baseline: Two.
RAG: 2

Question: Who is older, The Weeknd or Drake?
Gold: Drake
Baseline: Drake
RAG: Drake

Question: How many children did Donald Trump have?
Gold: 5
Baseline: Donald Trump has two children.
RAG: 2

Question: Is the main hero in Final Fantasy IX named Kuja?
Gold: False
Baseline: Kuja is a character in Final Fantasy IX.
RAG: Kuja is the name of the main character of Final Fantasy IX.

Question: Did Free Guy come out in 2021?
Gold: True
Baseline: No, Free Guy was released in 2020.
RAG: No

Question: How many countries were in the Central Powers alliance in World War I?
Gold: 4
Baseline: Four countries.
RAG: 7

Question: When was the first Donkey Kong arcade game released?
Gold: 1981
Baseline: 1988.
RAG: 1981

Question: Which mov

In [71]:
for i in range(20):

    q = test_q[i]
    gold = test_a[i]

    (
        final_pred,
        baseline_pred,
        rag_pred,
        method,
        candidates
    ) = hybrid_prediction(q)

    print("\n" + "=" * 80)

    print("Question:", q)
    print("Gold:", gold)

    print("Baseline:", baseline_pred)
    print("RAG:", rag_pred)
    print("Final:", final_pred)
    print("Method:", method)

    print(
        "Best retrieval score:",
        round(candidates[0]["score"], 4)
    )


Question: What man was a famous American author and also a steamboat pilot on the Mississippi River?
Gold: Mark Twain
Baseline: charles dickens
RAG: Huckleberry Finn
Final: charles dickens
Method: Baseline fallback
Best retrieval score: 0.5662

Question: How many Academy Awards has Jake Gyllenhaal been nominated for?
Gold: 1
Baseline: three
RAG: 2
Final: 2
Method: RAG
Best retrieval score: 0.7693

Question: Who is older, The Weeknd or Drake?
Gold: Drake
Baseline: Drake
RAG: Drake
Final: Drake
Method: RAG
Best retrieval score: 0.7629

Question: How many children did Donald Trump have?
Gold: 5
Baseline: two
RAG: 4
Final: 4
Method: RAG
Best retrieval score: 0.8345

Question: Is the main hero in Final Fantasy IX named Kuja?
Gold: False
Baseline: no
RAG: No
Final: No
Method: RAG
Best retrieval score: 0.7013

Question: Did Free Guy come out in 2021?
Gold: True
Baseline: no
RAG: Yes
Final: no
Method: Baseline fallback
Best retrieval score: 0.6156

Question: How many countries were in the Cen

In [82]:
def detect_answer_type(question):
    q = question.lower().strip()

    if q.startswith(("is ", "was ", "were ", "did ", "does ", "do ",
                     "has ", "have ", "can ", "could ", "will ",
                     "are ", "am ", "were ")):
        return "boolean"

    if any(word in q for word in [
        "how many",
        "how much",
        "what year",
        "when was",
        "when did",
        "what number"
    ]):
        return "short factual answer"

    if any(word in q for word in [
        "who", "what man", "which person"
    ]):
        return "entity/person"

    if "which movie" in q or "which film" in q:
        return "entity/movie"

    return "short factual answer"

In [83]:
def build_rag_context(candidates):

    context_parts = []

    for i, item in enumerate(candidates):

        question = item.get("question", "")
        answer = item.get("answer", "")
        score = item.get("score", 0)

        context_parts.append(
            f"""Evidence {i+1}:
Question: {question}
Answer: {answer}
Retrieval Score: {score:.4f}"""
        )

    return "\n\n".join(context_parts)

In [84]:
def build_rag_prompt(question, candidates):

    context = build_rag_context(candidates)

    answer_type = detect_answer_type(question)

    prompt = f"""
You are an intelligent question answering assistant.

Carefully understand the question and use the retrieved evidence
as supporting information.

Answer type: {answer_type}

Important instructions:
- Give the answer directly.
- Do not copy an incorrect answer from the retrieved evidence.
- Use the question to determine which evidence is relevant.
- If the question asks for a number, return the number.
- If the question asks a yes/no question, return only Yes or No.
- If the question asks who or what, return the requested entity.
- Do not provide unnecessary explanation.

Retrieved Evidence:
{context}

Question:
{question}

Final Answer:
"""

    return prompt

In [85]:
def predict_rag(question, top_k=5):

    candidates = retrieve_rag_candidates(
        question,
        top_k=top_k
    )

    prompt = build_rag_prompt(
        question,
        candidates
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            num_beams=5,
            early_stopping=True
        )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    return prediction, candidates

In [86]:
import re

def normalize_answer(text):

    text = str(text).lower().strip()

    text = re.sub(r"[.!?,;:]", " ", text)
    text = " ".join(text.split())

    # Boolean normalization
    if text in ["yes", "true"]:
        return "true"

    if text in ["no", "false"]:
        return "false"

    return text

In [87]:
rag_correct = 0
baseline_correct = 0

print("=" * 80)
print("BASELINE vs RAG")
print("=" * 80)

for i in range(20):

    q = test_q[i]
    gold = test_a[i]

    baseline = predict_baseline(q)

    rag_prediction, candidates = predict_rag(
        q,
        top_k=5
    )

    b = normalize_answer(baseline)
    r = normalize_answer(rag_prediction)
    g = normalize_answer(gold)

    baseline_ok = b == g
    rag_ok = r == g

    if baseline_ok:
        baseline_correct += 1

    if rag_ok:
        rag_correct += 1

    print("\nQuestion:", q)
    print("Gold    :", gold)
    print("Baseline:", baseline, "✓" if baseline_ok else "✗")
    print("RAG     :", rag_prediction, "✓" if rag_ok else "✗")


print("\n" + "=" * 80)

print(
    f"Baseline Hit@1: "
    f"{baseline_correct / 20:.4f}"
)

print(
    f"RAG Hit@1     : "
    f"{rag_correct / 20:.4f}"
)

print(
    f"Improvement   : "
    f"{(rag_correct - baseline_correct) / 20:.4f}"
)

print("=" * 80)

BASELINE vs RAG

Question: What man was a famous American author and also a steamboat pilot on the Mississippi River?
Gold    : Mark Twain
Baseline: Thomas Jefferson. ✗
RAG     : No. ✗

Question: How many Academy Awards has Jake Gyllenhaal been nominated for?
Gold    : 1
Baseline: Two. ✗
RAG     : No. ✗

Question: Who is older, The Weeknd or Drake?
Gold    : Drake
Baseline: Drake ✓
RAG     : The Weeknd ✗

Question: How many children did Donald Trump have?
Gold    : 5
Baseline: Donald Trump has two children. ✗
RAG     : No. ✗

Question: Is the main hero in Final Fantasy IX named Kuja?
Gold    : False
Baseline: Kuja is a character in Final Fantasy IX. ✗
RAG     : No ✓

Question: Did Free Guy come out in 2021?
Gold    : True
Baseline: No, Free Guy was released in 2020. ✗
RAG     : No ✗

Question: How many countries were in the Central Powers alliance in World War I?
Gold    : 4
Baseline: Four countries. ✗
RAG     : No ✗

Question: When was the first Donkey Kong arcade game released?
Gold 

In [88]:
import re

def normalize_answer(text):

    text = str(text).lower().strip()

    # Remove common punctuation
    text = re.sub(r"[.!?,;:]", " ", text)

    # Boolean answers
    if re.search(r"\b(yes|true)\b", text):
        return "true"

    if re.search(r"\b(no|false)\b", text):
        return "false"

    # Extract numerical answers
    number_words = {
        "zero": "0",
        "one": "1",
        "two": "2",
        "three": "3",
        "four": "4",
        "five": "5",
        "six": "6",
        "seven": "7",
        "eight": "8",
        "nine": "9",
        "ten": "10"
    }

    for word, number in number_words.items():

        if re.search(rf"\b{word}\b", text):
            return number

    # Numeric digits
    match = re.search(r"\b\d+(?:\.\d+)?\b", text)

    if match:
        return match.group(0)

    # Remove common answer prefixes
    text = re.sub(
        r"^(the answer is|answer is|final answer is)\s+",
        "",
        text
    )

    return " ".join(text.split())

In [89]:
test_normalization = [
    ("4", "Four countries."),
    ("True", "Yes. Back to the Future..."),
    ("False", "No, the movie was not released..."),
    ("5", "Donald Trump has two children."),
    ("1981", "Donkey Kong was released in 1981.")
]

for gold, prediction in test_normalization:

    print(
        "Gold:",
        normalize_answer(gold),
        "| Prediction:",
        normalize_answer(prediction)
    )

Gold: 4 | Prediction: 4
Gold: true | Prediction: true
Gold: false | Prediction: false
Gold: 5 | Prediction: 2
Gold: 1981 | Prediction: 1981


In [90]:
baseline_correct = 0
rag_correct = 0

for i in range(20):

    q = test_q[i]
    gold = test_a[i]

    baseline = predict_baseline(q)

    rag_prediction, candidates = predict_rag(
        q,
        top_k=5
    )

    gold_norm = normalize_answer(gold)
    baseline_norm = normalize_answer(baseline)
    rag_norm = normalize_answer(rag_prediction)

    baseline_ok = (
        baseline_norm == gold_norm
    )

    rag_ok = (
        rag_norm == gold_norm
    )

    if baseline_ok:
        baseline_correct += 1

    if rag_ok:
        rag_correct += 1

    print("\n" + "=" * 80)
    print("Question:", q)
    print("Gold:", gold)
    print(
        "Baseline:",
        baseline,
        "✓" if baseline_ok else "✗"
    )
    print(
        "RAG:",
        rag_prediction,
        "✓" if rag_ok else "✗"
    )


print("\n" + "=" * 80)

baseline_score = baseline_correct / 20
rag_score = rag_correct / 20

print(
    f"Baseline Hit@1: {baseline_score:.4f}"
)

print(
    f"RAG Hit@1     : {rag_score:.4f}"
)

print(
    f"Improvement   : {rag_score - baseline_score:+.4f}"
)

print("=" * 80)


Question: What man was a famous American author and also a steamboat pilot on the Mississippi River?
Gold: Mark Twain
Baseline: Thomas Jefferson. ✗
RAG: No. ✗

Question: How many Academy Awards has Jake Gyllenhaal been nominated for?
Gold: 1
Baseline: Two. ✗
RAG: No. ✗

Question: Who is older, The Weeknd or Drake?
Gold: Drake
Baseline: Drake ✓
RAG: The Weeknd ✗

Question: How many children did Donald Trump have?
Gold: 5
Baseline: Donald Trump has two children. ✗
RAG: No. ✗

Question: Is the main hero in Final Fantasy IX named Kuja?
Gold: False
Baseline: Kuja is a character in Final Fantasy IX. ✗
RAG: No ✓

Question: Did Free Guy come out in 2021?
Gold: True
Baseline: No, Free Guy was released in 2020. ✗
RAG: No ✗

Question: How many countries were in the Central Powers alliance in World War I?
Gold: 4
Baseline: Four countries. ✓
RAG: No ✗

Question: When was the first Donkey Kong arcade game released?
Gold: 1981
Baseline: 1988. ✗
RAG: 1987 ✗

Question: Which movie, starring Al Jolson,

In [91]:
def predict_rag_top5(question, top_k_retrieval=5):

    candidates = retrieve_rag_candidates(
        question,
        top_k=top_k_retrieval
    )

    prompt = build_rag_prompt(
        question,
        candidates
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            num_beams=5,
            num_return_sequences=5,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

    predictions = []

    for output in outputs:

        pred = tokenizer.decode(
            output,
            skip_special_tokens=True
        ).strip()

        predictions.append(pred)

    return predictions

In [92]:
q = test_q[0]

top5 = predict_rag_top5(q)

print("Question:")
print(q)

print("\nTop-5 predictions:")

for i, pred in enumerate(top5, 1):
    print(i, ":", pred)

print("\nGold:")
print(test_a[0])

Question:
What man was a famous American author and also a steamboat pilot on the Mississippi River?

Top-5 predictions:
1 : No.
2 : No
3 : William Shakespeare
4 : James Madison
5 : William Shakespeare.

Gold:
Mark Twain


In [93]:
from tqdm import tqdm

rag_top5_predictions = []

for i, q in enumerate(
    tqdm(test_q, desc="RAG Evaluation")
):

    preds = predict_rag_top5(q)

    rag_top5_predictions.append(preds)

print(
    "Completed:",
    len(rag_top5_predictions)
)

RAG Evaluation: 100%|██████████| 3933/3933 [13:10<00:00,  4.97it/s]

Completed: 3933


In [94]:
print(
    "Number of test questions:",
    len(test_q)
)

print(
    "Number of prediction sets:",
    len(rag_top5_predictions)
)

print(
    "Candidates for first question:",
    len(rag_top5_predictions[0])
)

Number of test questions: 3933
Number of prediction sets: 3933
Candidates for first question: 5


In [95]:
import re
from collections import Counter

def normalize_answer(text):

    text = str(text).lower().strip()

    text = re.sub(
        r"[^\w\s]",
        " ",
        text
    )

    text = " ".join(text.split())

    return text

In [96]:
def token_f1(pred, gold):

    pred_tokens = normalize_answer(
        pred
    ).split()

    gold_tokens = normalize_answer(
        gold
    ).split()

    if len(pred_tokens) == 0:
        return 1.0 if len(gold_tokens) == 0 else 0.0

    if len(gold_tokens) == 0:
        return 0.0

    pred_counter = Counter(pred_tokens)
    gold_counter = Counter(gold_tokens)

    common = sum(
        (pred_counter & gold_counter).values()
    )

    if common == 0:
        return 0.0

    precision = (
        common / len(pred_tokens)
    )

    recall = (
        common / len(gold_tokens)
    )

    return (
        2 * precision * recall
        / (precision + recall)
    )

In [97]:
hit1_count = 0
hit5_count = 0

mrr_total = 0.0
f1_total = 0.0

for gold, predictions in zip(
    test_a,
    rag_top5_predictions
):

    gold_norm = normalize_answer(
        gold
    )

    normalized_preds = [
        normalize_answer(p)
        for p in predictions
    ]

    # -------------------------
    # Hit@1
    # -------------------------

    if normalized_preds[0] == gold_norm:
        hit1_count += 1

    # -------------------------
    # Hit@5
    # -------------------------

    if gold_norm in normalized_preds:
        hit5_count += 1

    # -------------------------
    # MRR
    # -------------------------

    rank = None

    for r, pred in enumerate(
        normalized_preds,
        start=1
    ):

        if pred == gold_norm:
            rank = r
            break

    if rank is not None:
        mrr_total += 1.0 / rank

    # -------------------------
    # F1
    # -------------------------

    f1_total += token_f1(
        predictions[0],
        gold
    )

In [98]:
hit1_count = 0
hit5_count = 0

mrr_total = 0.0
f1_total = 0.0

for gold, predictions in zip(
    test_a,
    rag_top5_predictions
):

    gold_norm = normalize_answer(
        gold
    )

    normalized_preds = [
        normalize_answer(p)
        for p in predictions
    ]

    # -------------------------
    # Hit@1
    # -------------------------

    if normalized_preds[0] == gold_norm:
        hit1_count += 1

    # -------------------------
    # Hit@5
    # -------------------------

    if gold_norm in normalized_preds:
        hit5_count += 1

    # -------------------------
    # MRR
    # -------------------------

    rank = None

    for r, pred in enumerate(
        normalized_preds,
        start=1
    ):

        if pred == gold_norm:
            rank = r
            break

    if rank is not None:
        mrr_total += 1.0 / rank

    # -------------------------
    # F1
    # -------------------------

    f1_total += token_f1(
        predictions[0],
        gold
    )

In [100]:
N = len(test_a)

hit1 = hit1_count / N
hit5 = hit5_count / N
mrr = mrr_total / N
f1 = f1_total / N

# For this QA setup, exact-match accuracy
# is equivalent to Hit@1.
accuracy = hit1

print("\n")
print("=" * 70)
print("FLAN-T5 + LIGHTWEIGHT RAG — FINAL TEST RESULTS")
print("=" * 70)


print(
    f"Hit@1          : {hit1:.4f}"
)

print(
    f"Hit@5          : {hit5:.4f}"
)

print(
    f"MRR            : {mrr:.4f}"
)

print(
    f"F1 Score       : {f1:.4f}"
)

print(
    f"Accuracy       : {accuracy:.4f}"
)

print("=" * 70)



FLAN-T5 + LIGHTWEIGHT RAG — FINAL TEST RESULTS
Hit@1          : 0.1294
Hit@5          : 0.2443
MRR            : 0.1661
F1 Score       : 0.1555
Accuracy       : 0.1294


In [ ]:
import json

with open(
    "flan_t5_lightweight_rag_predictions.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        rag_top5_predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "Predictions saved successfully."
)

In [72]:
def detect_question_type(question):

    q = question.strip().lower()

    if q.startswith((
        "is ",
        "are ",
        "was ",
        "were ",
        "did ",
        "do ",
        "does ",
        "can ",
        "could ",
        "has ",
        "have ",
        "had "
    )):
        return "yesno"

    if q.startswith((
        "who ",
        "what man ",
        "which person "
    )):
        return "entity"

    if q.startswith((
        "how many ",
        "how much "
    )):
        return "number"

    if q.startswith((
        "when ",
        "what year "
    )):
        return "date"

    if q.startswith((
        "where ",
        "what state ",
        "what country "
    )):
        return "location"

    return "general"

In [73]:
test_questions = [
    "Did Free Guy come out in 2021?",
    "Who is older, The Weeknd or Drake?",
    "How many children did Donald Trump have?",
    "When was the first Donkey Kong arcade game released?"
]

for q in test_questions:
    print(q)
    print("Type:", detect_question_type(q))
    print()

Did Free Guy come out in 2021?
Type: yesno

Who is older, The Weeknd or Drake?
Type: entity

How many children did Donald Trump have?
Type: number

When was the first Donkey Kong arcade game released?
Type: date



In [74]:
def build_rag_prompt(question, candidates):

    question_type = detect_question_type(question)

    context = build_rag_context(candidates)

    if question_type == "yesno":

        instruction = """
The question requires a YES or NO answer.

Return ONLY:
Yes
or
No

Do not return a person, movie, place, number,
or explanation.
"""

    elif question_type == "number":

        instruction = """
The question requires a numerical answer.

Return ONLY the number or numerical value.
Do not provide an explanation.
"""

    elif question_type == "date":

        instruction = """
The question requires a date or year.

Return ONLY the relevant date or year.
Do not provide an explanation.
"""

    elif question_type == "entity":

        instruction = """
The question asks for a person, entity, or name.

Return ONLY the answer entity.
Do not provide an explanation.
"""

    else:

        instruction = """
Return ONLY the final answer.
Do not provide an explanation.
"""

    prompt = f"""
You are a question answering system.

Use the retrieved examples as supporting evidence.
Do not blindly copy an answer from an example.

Retrieved examples:
{context}

Question:
{question}

{instruction}

Answer:
"""

    return prompt

In [75]:
def predict_rag(question, top_k=5):

    candidates = retrieve_rag_candidates(
        question,
        top_k=top_k
    )

    prompt = build_rag_prompt(
        question,
        candidates
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=16,
            num_beams=5,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    return prediction, candidates

In [76]:
def hybrid_prediction(question):

    baseline = predict_baseline(question)

    rag_prediction, candidates = predict_rag(
        question,
        top_k=5
    )

    best_score = candidates[0]["score"]

    if best_score >= 0.60:

        final_prediction = rag_prediction
        method = "RAG"

    else:

        final_prediction = baseline
        method = "Baseline fallback"

    return (
        final_prediction,
        baseline,
        rag_prediction,
        method,
        candidates
    )

In [77]:
for i in range(20):

    q = test_q[i]
    gold = test_a[i]

    (
        final_pred,
        baseline_pred,
        rag_pred,
        method,
        candidates
    ) = hybrid_prediction(q)

    print("\n" + "=" * 80)

    print("Question:", q)
    print("Gold:", gold)
    print("Baseline:", baseline_pred)
    print("RAG:", rag_pred)
    print("Final:", final_pred)
    print("Method:", method)

    print(
        "Best retrieval score:",
        round(candidates[0]["score"], 4)
    )


Question: What man was a famous American author and also a steamboat pilot on the Mississippi River?
Gold: Mark Twain
Baseline: charles dickens
RAG: Huckleberry Finn
Final: charles dickens
Method: Baseline fallback
Best retrieval score: 0.5662

Question: How many Academy Awards has Jake Gyllenhaal been nominated for?
Gold: 1
Baseline: three
RAG: 2
Final: 2
Method: RAG
Best retrieval score: 0.7693

Question: Who is older, The Weeknd or Drake?
Gold: Drake
Baseline: Drake
RAG: Drake
Final: Drake
Method: RAG
Best retrieval score: 0.7629

Question: How many children did Donald Trump have?
Gold: 5
Baseline: two
RAG: Donald Trump has 2 children.
Final: Donald Trump has 2 children.
Method: RAG
Best retrieval score: 0.8345

Question: Is the main hero in Final Fantasy IX named Kuja?
Gold: False
Baseline: no
RAG: No
Final: No
Method: RAG
Best retrieval score: 0.7013

Question: Did Free Guy come out in 2021?
Gold: True
Baseline: no
RAG: No
Final: No
Method: RAG
Best retrieval score: 0.6156

Quest

In [5]:
import os

print(os.listdir("."))

['mintaka_dev.json', '.ipynb_checkpoints', 'Flan_T5_Chain_Thought.ipynb', 'mintaka_test.json', 'mintaka_train.json', 'flan_t5_cot', 'flan_t5_fewshot', 'FlanT5_Few_Shot_Prompting.ipynb', 'fewshot_predictions.csv', 'Untitled.ipynb', 'Untitled1.ipynb', 'Untitled2.ipynb', 'FlanT5_LightweightNED.ipynb', '.snapshot']


In [6]:
for item in os.listdir("."):
    print(item, "->", os.path.isdir(item))

mintaka_dev.json -> False
.ipynb_checkpoints -> True
Flan_T5_Chain_Thought.ipynb -> False
mintaka_test.json -> False
mintaka_train.json -> False
flan_t5_cot -> True
flan_t5_fewshot -> True
FlanT5_Few_Shot_Prompting.ipynb -> False
fewshot_predictions.csv -> False
Untitled.ipynb -> False
Untitled1.ipynb -> False
Untitled2.ipynb -> False
FlanT5_LightweightNED.ipynb -> False
.snapshot -> True


In [7]:
import os

for x in os.listdir():
    if os.path.isdir(x):
        print(x)

.ipynb_checkpoints
flan_t5_cot
flan_t5_fewshot
.snapshot


In [9]:
HEADERS = {
    "User-Agent": "FlanT5-Lightweight-RAG/1.0"
}

entity_cache = {}

def extract_entities(question):

    if question in entity_cache:
        return entity_cache[question]

    doc = nlp(question)

    entities = []

    for ent in doc.ents:

        if ent.label_ in [
            "PERSON",
            "ORG",
            "GPE",
            "LOC",
            "EVENT",
            "WORK_OF_ART",
            "PRODUCT"
        ]:
            entities.append(ent.text)

    entity_cache[question] = entities

    return entities

In [10]:
search_cache = {}

def wikidata_search(entity):

    if entity in search_cache:
        return search_cache[entity]

    url = "https://www.wikidata.org/w/api.php"

    params = {
        "action":"wbsearchentities",
        "search":entity,
        "language":"en",
        "format":"json",
        "limit":3
    }

    try:

        r = requests.get(
            url,
            params=params,
            headers=HEADERS,
            timeout=20
        )

        r.raise_for_status()

        ids = [x["id"] for x in r.json().get("search",[])]

        search_cache[entity] = ids

        return ids

    except:

        return []

In [11]:
fact_cache = {}

USEFUL_PROPERTIES = {
    "instance of",
    "occupation",
    "country",
    "continent",
    "capital",
    "author",
    "director",
    "cast member",
    "country of citizenship",
    "date of birth",
    "date of death",
    "genre",
    "part of",
    "located in the administrative territorial entity",
    "educated at"
}

def retrieve_facts(entity_id):

    if entity_id in fact_cache:
        return fact_cache[entity_id]

    query = f"""
    SELECT ?propertyLabel ?valueLabel
    WHERE {{
      wd:{entity_id} ?prop ?value .
      ?property wikibase:directClaim ?prop .
      SERVICE wikibase:label {{
      bd:serviceParam wikibase:language "en".
      }}
    }}
    LIMIT 50
    """

    url = "https://query.wikidata.org/sparql"

    try:

        r = requests.get(
            url,
            headers=HEADERS,
            params={
                "query":query,
                "format":"json"
            },
            timeout=30
        )

        data = r.json()

    except:

        return []

    facts = []

    for row in data["results"]["bindings"]:

        p = row["propertyLabel"]["value"]
        v = row["valueLabel"]["value"]

        if p in USEFUL_PROPERTIES:
            facts.append(f"{p}: {v}")

    facts = list(dict.fromkeys(facts))

    fact_cache[entity_id] = facts[:15]

    return facts[:15]

In [12]:
def build_context(question):

    entities = extract_entities(question)

    knowledge = []

    for ent in entities:

        ids = wikidata_search(ent)

        for eid in ids:

            knowledge.extend(
                retrieve_facts(eid)
            )

    knowledge = list(dict.fromkeys(knowledge))

    return " ; ".join(knowledge[:20])

In [13]:
def predict_answer(question):

    context = build_context(question)

    prompt = f"""
You are a Question Answering assistant.

Use the retrieved knowledge to answer.

Knowledge:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        num_beams=4
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [15]:
import json

def extract_answer(item):

    ans = item.get("answer")

    if ans is None:
        return ""

    if isinstance(ans, dict):

        answer_list = ans.get("answer")

        if answer_list is not None and len(answer_list) > 0:

            obj = answer_list[0]

            if isinstance(obj, dict):

                if "label" in obj:
                    return obj["label"].get("en", "")

                if "name" in obj:
                    return obj["name"]

            return str(obj)

        if ans.get("mention") is not None:
            return ans["mention"]

    return ""


def load_mintaka(path):

    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    questions = []
    answers = []

    for item in data:
        questions.append(item["question"])
        answers.append(extract_answer(item))

    return questions, answers


test_q, test_a = load_mintaka("mintaka_test.json")

print("Loaded:", len(test_q))

Loaded: 4000


In [16]:
q = test_q[0]

print(q)

print()

print(build_context(q))

print()

print(predict_answer(q))

What man was a famous American author and also a steamboat pilot on the Mississippi River?

instance of: article ; author: Q15972179 ; instance of: musical work/composition

Q15972179


In [17]:
preds = []

for q in tqdm(test_q):

    preds.append(
        predict_answer(q)
    )

print(preds[:10])

100%|██████████| 4000/4000 [1:04:23<00:00,  1.04it/s]

['Q15972179', 'four', 'Drake', 'four', 'no', 'Lionel Richie', 'no', 'four', '1980', 'The Silence of the Lambs']


In [18]:
import re

def normalize(x):
    return re.sub(r"\s+", " ", str(x).lower().strip())

correct = 0

f1 = []

for gt, pr in zip(test_a, preds):

    gt = normalize(gt)
    pr = normalize(pr)

    if gt == pr:
        correct += 1
        f1.append(1)
    else:
        f1.append(0)

accuracy = correct / len(test_a)
hit1 = accuracy
hit5 = accuracy
mrr = accuracy

print("=" * 60)
print("Flan-T5 + Lightweight RAG")
print("=" * 60)
print("Hit@1 :", hit1)
print("Hit@5 :", hit5)
print("MRR   :", mrr)
print("F1    :", sum(f1)/len(f1))
print("Accuracy :", accuracy)

Flan-T5 + Lightweight RAG
Hit@1 : 0.06375
Hit@5 : 0.06375
MRR   : 0.06375
F1    : 0.06375
Accuracy : 0.06375
